# Text2Cypher on the TalentBridge graph — Notebook 01: Setup and graph tour

**Goal of the series.** Let a recruiter ask questions in plain English ("Which candidates claim ML?") and have a *local* LLM served by **Ollama** translate them into **Cypher**, run them on **Neo4j**, and answer.

| Notebook | Topic | Key idea |
|---|---|---|
| **01** (this one) | Setup, loading the graph, schema extraction | The LLM can only be as good as the schema description we give it |
| 02 | Zero-shot Text2Cypher | Prompt anatomy, calling Ollama, extracting Cypher |
| 03 | Making it robust | Safety guard, entity linking, few-shot examples, self-correction |
| 04 | Evaluation | Execution accuracy on a benchmark, ablations, error analysis |
| 05 | Graph QA assistant | From rows to grounded answers, routing to trusted queries, prompt injection |

**What you need before starting**
1. Neo4j 5.x running (Desktop, Docker `docker run -p7474:7474 -p7687:7687 -e NEO4J_AUTH=neo4j/password neo4j:5`, or Aura). **Use an empty / scratch database: this notebook wipes it.**
2. Ollama installed and running (`ollama serve`), with at least one model pulled, e.g. `ollama pull qwen2.5-coder:7b` and `ollama pull nomic-embed-text`.
3. This folder: `t2c.py`, `benchmark.json`, `examples.json` and `cypher/`.

## The pipeline we are going to build

```
 Natural-language question
          │
          ▼
 ┌──────────────────────┐   schema + business rules + (examples) + (entity hints)
 │  Prompt construction │◄───────────────────────────────────────────────────────
 └──────────┬───────────┘
            ▼
 ┌──────────────────────┐
 │  LLM (Ollama, local) │   e.g. qwen2.5-coder:7b
 └──────────┬───────────┘
            ▼  raw text
 ┌──────────────────────┐
 │  Extract Cypher      │   strip code fences / <think> blocks
 └──────────┬───────────┘
            ▼
 ┌──────────────────────┐   read-only guard ─► EXPLAIN ─► READ transaction
 │  Validate & execute  │──── error? ──► feed the error back to the LLM (retry)
 └──────────┬───────────┘
            ▼  rows
 ┌──────────────────────┐
 │  (Answer generation) │   LLM writes a sentence grounded ONLY in the rows
 └──────────────────────┘
```

### Step 1 — Install the Python packages
**What it does:** installs the Neo4j driver, the Ollama client, and pandas/matplotlib for tables and charts.
**Why:** `neo4j` talks the Bolt protocol to the database; `ollama` is a thin HTTP client for the local model server (port 11434). Nothing is sent to a cloud service.

In [ ]:
%pip install -q neo4j ollama pandas matplotlib

### Step 2 — Configuration
**What it does:** collects every setting (database credentials, Ollama address, model names) in one place.
**Why:** students can switch model (`T2C_MODEL=llama3.1:8b`) or database without touching the code. Change the password here or set the environment variable before launching Jupyter.

In [ ]:
import os, sys, json, re, time, math
from pathlib import Path

# --- Connection settings (override with environment variables if you prefer) ---
NEO4J_URI      = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")     # <- change me
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE") or None          # None = default database

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
MODEL       = os.getenv("T2C_MODEL", "qwen2.5-coder:7b")      # any chat model you pulled with `ollama pull`
EMBED_MODEL = os.getenv("T2C_EMBED_MODEL", "nomic-embed-text")

print("Neo4j :", NEO4J_URI, "| Ollama:", OLLAMA_HOST, "| model:", MODEL)

### Step 3 — A small wrapper around the Neo4j driver
**What it does:** defines `Neo4jGraph` with four methods:
* `run_read` runs a query inside a **read transaction** with a timeout and returns a list of Python dicts (one per row),
* `run_write` is used only by the loader,
* `explain` asks Neo4j to *compile* a query without running it (we will use it to catch syntax errors before execution),
* `close` releases the connection.

**Why it matters for Text2Cypher:** generated queries are untrusted code. Running them in a `READ_ACCESS` session means the **database itself** refuses any write, even if our own checks miss something. The timeout protects against accidental cartesian products.

*Ask your students:* why is `@unit_of_work(timeout=...)` better than trusting the LLM to add `LIMIT`?

In [ ]:
class Neo4jGraph:
    """Thin wrapper around the official neo4j driver.

    * run_read  -> executes in a READ transaction (the database itself refuses writes)
    * run_write -> used only by the loader
    * explain   -> compiles a query without running it (syntax/schema check)
    """

    def __init__(self, uri: str, user: str, password: str, database: str | None = None):
        from neo4j import GraphDatabase
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        self.driver.verify_connectivity()
        self.database = database

    def run_read(self, cypher: str, params: dict | None = None, timeout: float = 10.0) -> list[dict]:
        from neo4j import READ_ACCESS, unit_of_work

        @unit_of_work(timeout=timeout)
        def work(tx):
            return [record.data() for record in tx.run(cypher, params or {})]

        with self.driver.session(database=self.database, default_access_mode=READ_ACCESS) as session:
            return session.execute_read(work)

    def run_write(self, cypher: str, params: dict | None = None) -> None:
        with self.driver.session(database=self.database) as session:
            session.run(cypher, params or {}).consume()

    def explain(self, cypher: str) -> None:
        """Raises a neo4j exception if the query does not compile."""
        from neo4j import READ_ACCESS
        with self.driver.session(database=self.database, default_access_mode=READ_ACCESS) as session:
            session.run("EXPLAIN " + cypher).consume()

    def close(self) -> None:
        self.driver.close()

### Step 4 — Loading the TalentBridge scripts
**What it does:** `split_cypher_script` cuts a `.cypher` file into individual statements (the driver runs one statement at a time); `load_talentbridge` wipes the database and runs, in order, the constraints (`01`), the data (`02`) and the change-request evolution with skill assessments (`05`).
**Why:** a reproducible, known dataset is essential: our evaluation in notebook 04 compares the LLM's answers with *gold* answers computed on exactly this data. Statements containing `$parameters` are example queries, not data, so they are skipped.

In [ ]:
def split_cypher_script(text: str) -> list[str]:
    """Split a .cypher file into statements (on ';' at end of line), dropping comment lines."""
    statements = []
    for chunk in re.split(r";\s*\n", text):
        body = "\n".join(l for l in chunk.splitlines() if not l.strip().startswith("//")).strip()
        if body:
            statements.append(body.rstrip(";"))
    return statements

def load_talentbridge(graph, cypher_dir: str | Path = "cypher", wipe: bool = True) -> dict:
    """Wipe the database (optional) and run 01, 02 and 05 scripts. Returns node/relationship counts."""
    cypher_dir = Path(cypher_dir)
    if wipe:
        graph.run_write("MATCH (n) DETACH DELETE n")
    for name in ["01_constraints_indexes.cypher", "02_load_data.cypher", "05_change_request.cypher"]:
        for stmt in split_cypher_script((cypher_dir / name).read_text(encoding="utf-8")):
            if "$" in stmt:          # parameterised example queries are not part of the load
                continue
            graph.run_write(stmt)
    nodes = graph.run_read("MATCH (n) RETURN count(n) AS n")[0]["n"]
    rels = graph.run_read("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"]
    return {"nodes": nodes, "relationships": rels}

### Step 5 — Connect and load
**What it does:** opens the connection (fails fast with a clear error if Neo4j is not reachable) and loads the data.
**Expected output:** about **180 nodes and 346 relationships**. If you rerun the cell you get the same numbers: the scripts use `MERGE` on stable IDs, so they are idempotent.

⚠️ `LOAD_DATA = True` deletes everything in the target database first.

In [ ]:
graph = Neo4jGraph(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)

LOAD_DATA = True          # set to False if the graph is already loaded
if LOAD_DATA:
    print(load_talentbridge(graph, "cypher", wipe=True))

### Step 6 — What is in the graph?
**What it does:** counts nodes per label.
**Why:** before asking an LLM to query a graph, *we* should know it. Note the reified nodes (`SkillClaim`, `WorkExperience`, `Application`, `ApplicationEvent`, `SkillAssessment`): they will be the main source of LLM mistakes, because a model's "intuition" is to link `Candidate` directly to `Skill`.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)

counts = graph.run_read("""
MATCH (n) UNWIND labels(n) AS label
RETURN label, count(*) AS nodes ORDER BY nodes DESC
""")
pd.DataFrame(counts)

### Step 7 — Look at one real path
**What it does:** shows candidate C01's skill claims with level, verification status and the evidence attached to each claim.
**Why:** this is the pattern `(Candidate)-[:MAKES_CLAIM]->(SkillClaim)-[:ABOUT_SKILL]->(Skill)` that every skill question needs. Show it on the Neo4j Browser too (`MATCH p=(:Candidate {candidateId:'C01'})-[:MAKES_CLAIM]->()-[*1]->() RETURN p`).

In [ ]:
pd.DataFrame(graph.run_read("""
MATCH (c:Candidate {candidateId: 'C01'})-[:MAKES_CLAIM]->(cl:SkillClaim)-[:ABOUT_SKILL]->(s:Skill)
OPTIONAL MATCH (cl)-[:EVIDENCED_BY]->(e)
RETURN c.name AS candidate, s.name AS skill, cl.level AS level,
       cl.verificationStatus AS status, collect(coalesce(e.experienceId, e.projectId)) AS evidence
ORDER BY level DESC
"""))

### Step 8 — Extracting the schema automatically
An LLM cannot see the database. Everything it knows about labels, relationship **directions** and property **names/values** comes from the text we put in the prompt. We build that text from the live graph with three plain Cypher queries (no APOC needed):

1. `NODE_PROPS_QUERY` – for every label and property: number of distinct values and up to 10 samples;
2. `REL_PATTERNS_QUERY` – every `(:A)-[:TYPE]->(:B)` combination that really exists;
3. `REL_PROPS_QUERY` – properties of relationships (e.g. `REQUIRES_SKILL.priority`).

`_describe_property` then decides how to describe each property: low-cardinality strings are listed as **enumerations** (`priority: one of ['MANDATORY', 'PREFERRED']`) because LLMs otherwise guess `'mandatory'` or `true`; IDs get one or two examples so the model learns the format (`'C01'`).

*Ask your students:* what could go wrong if we listed **all** values of every property? (prompt size, privacy: names of real candidates would leak into the prompt).

In [ ]:
NODE_PROPS_QUERY = """
MATCH (n) UNWIND labels(n) AS label UNWIND keys(n) AS key
WITH label, key, n[key] AS v
RETURN label, key, count(DISTINCT v) AS distinctCount, collect(DISTINCT v)[..10] AS samples
ORDER BY label, key
"""

REL_PATTERNS_QUERY = """
MATCH (a)-[r]->(b) UNWIND labels(a) AS fromLabel UNWIND labels(b) AS toLabel
RETURN DISTINCT fromLabel, type(r) AS relType, toLabel
ORDER BY relType, fromLabel, toLabel
"""

REL_PROPS_QUERY = """
MATCH ()-[r]->() UNWIND keys(r) AS key
WITH type(r) AS relType, key, r[key] AS v
RETURN relType, key, count(DISTINCT v) AS distinctCount, collect(DISTINCT v)[..10] AS samples
ORDER BY relType, key
"""

FREE_TEXT_KEYS = {'name', 'note', 'roleTitle', 'key', 'reason', 'title', 'label'}

def infer_type(value) -> str:
    """Map a Python / neo4j value to a Cypher type name."""
    cls = type(value).__name__.lower()
    if isinstance(value, bool):
        return "BOOLEAN"
    if isinstance(value, int):
        return "INTEGER"
    if isinstance(value, float):
        return "FLOAT"
    if isinstance(value, str):
        return "STRING"
    if isinstance(value, list):
        return "LIST"
    if cls == "datetime":
        return "DATETIME"
    if cls == "date":
        return "DATE"
    return cls.upper()

def _describe_property(key: str, distinct: int, samples: list) -> str:
    samples = [s for s in samples if s is not None]
    if not samples:
        return f"{key}: UNKNOWN"
    ptype = infer_type(samples[0])
    is_id = key.endswith("Id") or key == "key"
    if ptype in ("STRING", "INTEGER") and not is_id and key not in FREE_TEXT_KEYS and distinct <= 8:
        values = sorted(samples, key=str)
        return f"{key}: {ptype} one of {values!r}"
    if ptype in ("STRING", "DATE", "DATETIME"):
        examples = ", ".join(repr(str(s)) for s in samples[:2])
        return f"{key}: {ptype} e.g. {examples}"
    return f"{key}: {ptype}"

def extract_schema(graph) -> dict:
    """Introspect the live graph: labels+properties, relationship patterns, relationship properties."""
    nodes: dict[str, list[str]] = {}
    for row in graph.run_read(NODE_PROPS_QUERY):
        nodes.setdefault(row["label"], []).append(
            _describe_property(row["key"], row["distinctCount"], row["samples"]))
    rel_props: dict[str, list[str]] = {}
    for row in graph.run_read(REL_PROPS_QUERY):
        rel_props.setdefault(row["relType"], []).append(
            _describe_property(row["key"], row["distinctCount"], row["samples"]))
    patterns = [(r["fromLabel"], r["relType"], r["toLabel"]) for r in graph.run_read(REL_PATTERNS_QUERY)]
    return {"nodes": nodes, "rel_props": rel_props, "patterns": patterns}

def schema_to_text(schema: dict) -> str:
    """Render the schema in a compact, Cypher-like notation that LLMs read well."""
    lines = ["Node labels and properties:"]
    for label, props in schema["nodes"].items():
        lines.append(f"(:{label} {{{', '.join(props)}}})")
    lines.append("")
    lines.append("Relationship patterns (direction matters):")
    for a, rel, b in schema["patterns"]:
        props = schema["rel_props"].get(rel)
        prop_txt = f" {{{', '.join(props)}}}" if props else ""
        lines.append(f"(:{a})-[:{rel}{prop_txt}]->(:{b})")
    return "\n".join(lines)

### Step 9 — The schema as the LLM will see it
**What it does:** prints the schema text.
**Discuss with students:** read it as if you were the model. What does it **not** tell you?
* that "ML" and "Machine Learning" are the same skill (aliases),
* that a requirement is *met* only by a **VERIFIED** claim with level ≥ `minLevel`,
* that `endDate = null` means "still ongoing",
* that two people are called "Amina Idrissi".

Those are **business rules**; notebook 02 adds them explicitly to the prompt. A rough size estimate is ~4 characters per token, so this schema costs ≈ 1,300 tokens on every call.

In [ ]:
schema = extract_schema(graph)
schema_text = schema_to_text(schema)
print(schema_text)
print(f"\n≈ {len(schema_text)//4} tokens")

### Step 10 — Is Ollama ready?
**What it does:** lists the models installed in your local Ollama and sends a one-line test prompt.
**If it fails:** start the server (`ollama serve`) and pull a model (`ollama pull qwen2.5-coder:7b`).

**Choosing a model (rules of thumb for a laptop):**

| Model (Ollama tag) | Size | Notes |
|---|---|---|
| `qwen2.5-coder:7b` | ~4.7 GB | strong at code/Cypher for its size — our default |
| `llama3.1:8b` | ~4.9 GB | good general model, sometimes adds prose around the query |
| `mistral:7b` | ~4.1 GB | fast, weaker on multi-hop patterns |
| `qwen2.5-coder:14b` / `:32b` | 9 / 20 GB | noticeably better on hard questions if you have the RAM/GPU |
| reasoning models (`qwen3`, `deepseek-r1`) | varies | emit `<think>…</think>` first; our extractor strips it |

In [ ]:
def list_ollama_models(host: str | None = None) -> list[str]:
    from ollama import Client
    client = Client(host=host or os.getenv("OLLAMA_HOST", "http://localhost:11434"))
    response = client.list()
    models = response["models"] if isinstance(response, dict) else response.models
    return [getattr(m, "model", None) or m.get("model") or m.get("name") for m in models]

available = list_ollama_models(OLLAMA_HOST)
print("Installed models:", available)
if MODEL not in available:
    print(f"\n⚠️  {MODEL} is not installed. Run:  ollama pull {MODEL}")

In [ ]:
from ollama import Client
client = Client(host=OLLAMA_HOST)
reply = client.chat(model=MODEL, messages=[{"role": "user", "content": "In one sentence: what is Cypher?"}],
                    options={"temperature": 0})
print(reply["message"]["content"])

## Exercises
1. Modify `_describe_property` so that `level` is described as `INTEGER 1=beginner, 2=intermediate, 3=advanced`. Is this better in the schema or in the business rules?
2. The schema lists `(:ApplicationEvent)-[:PERFORMED_BY]->(:Assessor)` because recruiter R02 is also an assessor (multi-label node). Is this pattern useful or noise for the LLM? How would you filter it?
3. Measure the schema size in tokens for a graph with 50 labels. Propose a strategy to send only the *relevant part* of the schema for a question.